# 01 — Ingest Landing

**Purpose:** Validate and supply the **landing zone** with raw NYC yellow taxi Parquet files (January–May 2023).

| Item | Value |
|------|--------|
| Catalog | `workspace` |
| Schema | `nyc_taxi` |
| Volume | `landing` |
| Layer | Raw (unchanged source files) |

> **Free Edition:** Upload files manually to the landing Volume. Automated download from TLC URLs is commented out (network often blocked).

---

## Configuration

Set the catalog name if `spark.catalog.listCatalogs()` shows something other than `workspace`.

---

In [0]:
# List available catalogs (PySpark Catalog API)
display(
    spark.createDataFrame(
        [(c.name,) for c in spark.catalog.listCatalogs()],
        ["catalog"],
    )
)

## Setup — imports and variables

Single source of truth for catalog, paths, and months. Run this cell before any code that uses `months` or `landing_base`.

---

In [0]:
# --- Imports ---
from functools import reduce

from pyspark.sql import DataFrame
from pyspark.sql.functions import col

# --- Config ---
CATALOG = "workspace"
SCHEMA = "nyc_taxi"
months = ["01", "02", "03", "04", "05"]
base_url = "https://d37ci6vzurychd.cloudfront.net/trip-data"
landing_base = f"/Volumes/{CATALOG}/{SCHEMA}/landing/yellow_taxi/2023"

## Create schema and volumes

Creates Unity Catalog objects if they do not exist:

- **`landing`** — raw Parquet files from TLC
- **`consumption`** — curated Delta files (used in notebook 02)

---

In [0]:
# --- Catalog, schema, and volumes (landing + consumption) ---
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.landing")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.consumption")
display(spark.sql(f"SHOW VOLUMES IN {SCHEMA}"))

## Validate landing files

Checks that all **5 monthly Parquet files** exist under the expected folder structure:

```text
/Volumes/workspace/nyc_taxi/landing/yellow_taxi/2023/
├── 01/yellow_tripdata_2023-01.parquet
├── 02/yellow_tripdata_2023-02.parquet
├── ...
└── 05/yellow_tripdata_2023-05.parquet
```

If any file is missing, upload it via **Catalog → Volume → Upload** before continuing.

---

In [0]:
# --- Free Edition: files must already exist in the landing Volume (manual upload) ---
# Expected layout per month:
expected_files = [
    f"{landing_base}/{m}/yellow_tripdata_2023-{m}.parquet"
    for m in months
]
missing = []
for path in expected_files:
    try:
        dbutils.fs.ls(path)
        print(f"OK  {path}")
    except Exception as e:
        print(f"MISSING  {path}")
        missing.append(path)
if missing:
    raise Exception(
        f"{len(missing)} file(s) missing in landing. "
        "Upload Parquet files to the 'landing' Volume before continuing."
    )
else:
    print("Landing complete: all 5 files found.")


## (Optional) Automated ingestion from URL

> **Illustrative / Enterprise.** Usually **fails on Free Edition** (`UnknownHostException`).

Flow when network allows:

1. `urllib` downloads to `/tmp` on the driver
2. `dbutils.fs.cp` copies to the landing path

**Free Edition:** use manual upload instead of this cell.

**Enterprise:** change `landing_base` to cloud storage, e.g. `s3://my-company-datalake/raw/nyc_taxi/yellow_taxi/2023`.

---

In [0]:
# --- (Optional) Automated ingestion from URL — only if workspace network allows it ---
# Usually fails on Free Edition (UnknownHostException).
# import urllib.request
# base_url = "https://d37ci6vzurychd.cloudfront.net/trip-data"
# for m in months:
#     url = f"{base_url}/yellow_tripdata_2023-{m}.parquet"
#     local_path = f"/tmp/yellow_tripdata_2023-{m}.parquet"
#     dest = f"{landing_base}/{m}/yellow_tripdata_2023-{m}.parquet"
#     urllib.request.urlretrieve(url, local_path)
#     dbutils.fs.mkdirs(f"{landing_base}/{m}")
#     dbutils.fs.cp(f"file:{local_path}", dest)

## Quick validation — row counts per month

Reads each landing file with Spark to confirm files are readable and prints row/column counts.  
Requires the **Setup** cell above (`months`, `landing_base`, imports).  
Does **not** write to the consumption layer.

---

In [0]:
# Schema unification function
def read_month(path: str) -> DataFrame:
    return (
        spark.read.parquet(path) 
        .select(
            col("VendorID").cast("long"),
            col("passenger_count").cast("long"),
            col("total_amount").cast("double"),
            col("tpep_pickup_datetime").cast("timestamp"),
            col("tpep_dropoff_datetime").cast("timestamp"),
        )
    )

dfs = []
for m in months:
    path = f"{landing_base}/{m}/yellow_tripdata_2023-{m}.parquet"
    print(f"Reading {path}")
    df = read_month(path)
    print(f"  rows: {df.count()}")
    dfs.append(df)

df_landing = reduce(DataFrame.unionByName, dfs)
display(df_landing.limit(5))
print("Total:", df_landing.count())

## Landing complete

Next step: run **`02_landing_to_consumption`** to cleanse data and write the consumption Delta layer.

---